# AzSemBench — PAWS Fine-Tuning

Fine-tunes XLM-RoBERTa, multilingual BERT, multilingual DistilBERT and mDeBERTa as cross-encoders. Best checkpoint is selected by dev F1; neural decisions use probability >= 0.5. Lexical thresholds use dev F1. Test is held out. Supplied results for all four models are included in `results/paws_finetuned/paws_finetuned_all_metrics.csv`; see the README for the held-out test comparison.

Run top to bottom in Colab, Kaggle (Internet + GPU enabled), or local Jupyter. Outputs are written under `outputs/`. See the repository README for metrics, supplied results and dataset filtering.


## 1. Install dependencies

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.12,<2" "scikit-learn>=1.4,<2" "sentence-transformers>=3.4,<6" "transformers>=4.48,<5" "datasets>=3,<5" "accelerate>=1,<2" "torch>=2.4,<3" "sentencepiece>=0.2,<1" "protobuf>=4,<7" "rapidfuzz>=3,<4" "python-dotenv>=1,<2"


If Kaggle still shows a tokenizer/backend error immediately after installing dependencies, restart the Kaggle session once and rerun from the top.


## 2. Imports and configuration

In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import gc, inspect, os, re, shutil, time, warnings

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from rapidfuzz.distance import Levenshtein, JaroWinkler
from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score,
    average_precision_score, brier_score_loss, confusion_matrix
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
    EarlyStoppingCallback, set_seed
)

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
MODELS = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-cased",
    "distilbert/distilbert-base-multilingual-cased",
    "microsoft/mdeberta-v3-base",
]

SEED = 42
MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 4
EARLY_STOPPING_PATIENCE = 1

OUTPUT_DIR = Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs/paws_finetuned"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)

print("Output:", OUTPUT_DIR)
for m in MODELS:
    print(" -", m)


## 3. Load `core_model_ready.csv` and isolate PAWS

In [ ]:
# Configuration: environment variables override these defaults.
import os
from pathlib import Path
from urllib.request import urlretrieve
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")  # use a commit SHA for a fixed dataset
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
DATA_DIR = Path("data/model_ready")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "core_model_ready.csv"
# Download on each run so data is explicitly tied to REPO/REF.
try:
    urlretrieve(DATA_URL, CSV_PATH)
except Exception as exc:
    raise RuntimeError(f"Cannot download {DATA_URL}. Publish the prepared repo first and enable Internet.") from exc
import hashlib
print("Dataset:", DATA_URL)
print("SHA256:", hashlib.sha256(CSV_PATH.read_bytes()).hexdigest())

def clean(x):
    return "" if x is None or pd.isna(x) else str(x).strip()

csv_path = CSV_PATH
print("Dataset:", csv_path)

df = pd.read_csv(
    csv_path,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)

if "model_ready" in df.columns:
    ready = df["model_ready"].astype(str).str.lower().str.strip()
    df = df[ready.isin(["true", "1", "yes"])].copy()

mask = pd.Series(False, index=df.index)

if "task" in df.columns:
    mask |= df["task"].astype(str).str.lower().str.strip().eq("paraphrase_detection")

if "recomputed_task" in df.columns:
    mask |= df["recomputed_task"].astype(str).str.lower().str.strip().eq("paraphrase_detection")

if "source_dataset" in df.columns:
    mask |= df["source_dataset"].astype(str).str.lower().str.contains("paws", regex=False)

paws = df.loc[mask].copy()

required = ["sentence1_az", "sentence2_az", "label", "split"]
missing = [c for c in required if c not in paws.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

paws["sentence1_az"] = paws["sentence1_az"].map(clean)
paws["sentence2_az"] = paws["sentence2_az"].map(clean)
paws["label"] = pd.to_numeric(paws["label"], errors="raise")
if not paws["label"].isin([0, 1]).all(): raise ValueError("Paraphrase labels must be binary.")
paws["label"] = paws["label"].astype(int)
paws["split"] = paws["split"].astype(str).str.lower().str.strip()

paws = paws[
    (paws["sentence1_az"] != "") &
    (paws["sentence2_az"] != "")
].copy()

train_df = paws[paws["split"].eq("train")].copy()
dev_df   = paws[paws["split"].eq("dev")].copy()
test_df  = paws[paws["split"].eq("test")].copy()

print("\nSplit sizes:")
print(paws["split"].value_counts())

print("\nLabel counts:")
print(paws["label"].value_counts().sort_index())

for name, frame in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    if frame.empty:
        raise RuntimeError(f"{name} split is empty.")


## 4. Evaluation metrics

In [ ]:
def corr3(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    if len(y) < 2 or np.std(y) == 0 or np.std(p) == 0:
        return np.nan, np.nan, np.nan
    return (
        float(pearsonr(y, p).statistic),
        float(spearmanr(y, p).statistic),
        float(kendalltau(y, p).statistic),
    )

def probs(logits):
    x = np.asarray(logits, float)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def cls_metrics(y, prob, thr=0.5):
    y = np.asarray(y, int)
    prob = np.asarray(prob, float)
    pred = (prob >= thr).astype(int)

    cm = confusion_matrix(y, pred, labels=[0, 1])
    pearson, spearman, kendall = corr3(y, prob)

    d = {
        "threshold": float(thr),
        "accuracy": float(accuracy_score(y, pred)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "brier_score": float(brier_score_loss(y, prob)),
        "pearson": pearson,
        "spearman": spearman,
        "kendall_tau": kendall,
        "tn": int(cm[0,0]),
        "fp": int(cm[0,1]),
        "fn": int(cm[1,0]),
        "tp": int(cm[1,1]),
    }

    if len(np.unique(y)) == 2:
        d["roc_auc"] = float(roc_auc_score(y, prob))
        d["average_precision"] = float(average_precision_score(y, prob))
    else:
        d["roc_auc"] = np.nan
        d["average_precision"] = np.nan

    return d

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return cls_metrics(labels, probs(logits)[:,1], 0.5)


Metrics saved for each model/split:

- Accuracy, precision, recall, F1
- Balanced accuracy
- Matthews correlation coefficient (MCC)
- ROC-AUC
- Average precision
- Brier score
- Pearson, Spearman, Kendall τ
- TN, FP, FN, TP


## 5. Compatibility helpers

In [ ]:
def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "__", x)

def load_tokenizer_safely(model_name):
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        print("Tokenizer: fast")
        return tok
    except Exception as e:
        print("Fast tokenizer failed. Falling back to slow tokenizer.")
        print("Reason:", str(e)[:400])
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        print("Tokenizer: slow")
        return tok

def make_ds(frame, tok):
    ds = Dataset.from_pandas(
        frame[["sentence1_az", "sentence2_az", "label"]].reset_index(drop=True)
    )
    ds = ds.map(
        lambda b: tok(
            b["sentence1_az"],
            b["sentence2_az"],
            truncation=True,
            max_length=MAX_LENGTH
        ),
        batched=True,
        desc="Tokenizing"
    )
    ds = ds.rename_column("label", "labels")
    keep = {"input_ids", "attention_mask", "token_type_ids", "labels"}
    return ds.remove_columns([c for c in ds.column_names if c not in keep])

def training_args(out):
    kw = dict(
        output_dir=str(out),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=torch.cuda.is_available(),
    )

    valid = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in valid:
        kw["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in valid:
        kw["evaluation_strategy"] = "epoch"

    kw = {k:v for k,v in kw.items() if k in valid}
    return TrainingArguments(**kw)

def build_trainer(model, tok, out, tr, dv):
    kwargs = dict(
        model=model,
        args=training_args(out),
        train_dataset=tr,
        eval_dataset=dv,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )]
    )

    trainer_params = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_params:
        kwargs["processing_class"] = tok
    elif "tokenizer" in trainer_params:
        kwargs["tokenizer"] = tok

    return Trainer(**kwargs)

def delete_checkpoints(model_out):
    deleted = []
    for p in model_out.glob("checkpoint-*"):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
            deleted.append(str(p))
    return deleted


## 6. Fine-tune all four multilingual models

In [ ]:
rows = []
predframes = []
summary = []

for model_name in MODELS:
    print("\n" + "="*100)
    print("MODEL:", model_name)
    print("="*100)

    out = OUTPUT_DIR / safe_name(model_name)
    out.mkdir(parents=True, exist_ok=True)

    tok = load_tokenizer_safely(model_name)

    tr = make_ds(train_df, tok)
    dv = make_ds(dev_df, tok)
    te = make_ds(test_df, tok)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    trainer = build_trainer(model, tok, out, tr, dv)

    t0 = time.time()
    result = trainer.train()

    best_model_dir = out / "best_model"
    trainer.save_model(best_model_dir)
    tok.save_pretrained(best_model_dir)

    for split_name, ds, frame in [
        ("train", tr, train_df),
        ("dev", dv, dev_df),
        ("test", te, test_df),
    ]:
        p = probs(trainer.predict(ds).predictions)[:,1]

        rows.append({
            "model": model_name,
            "split": split_name,
            "n": len(frame),
            **cls_metrics(frame["label"].values, p, 0.5)
        })

        z = frame[["sentence1_az", "sentence2_az", "label", "split"]].copy()
        if "source_id" in frame.columns:
            z.insert(0, "source_id", frame["source_id"].astype(str).values)

        z.insert(0, "model", model_name)
        z["p_paraphrase"] = p
        z["predicted_label"] = (p >= 0.5).astype(int)
        predframes.append(z)

    elapsed = time.time() - t0

    summary.append({
        "model": model_name,
        "best_dev_f1": trainer.state.best_metric,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "train_seconds": elapsed,
        "train_loss": float(result.training_loss)
    })

    deleted = delete_checkpoints(out)
    print("Deleted temporary checkpoints:", len(deleted))

    # Save progress after every model.
    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "paws_finetuned_all_metrics.csv",
        index=False
    )
    pd.concat(predframes, ignore_index=True).to_csv(
        OUTPUT_DIR / "paws_finetuned_predictions.csv",
        index=False
    )
    pd.DataFrame(summary).to_csv(
        OUTPUT_DIR / "paws_training_summary.csv",
        index=False
    )

    del trainer, model, tok, tr, dv, te
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Finished in {elapsed/60:.1f} min")


## 7. Show and save final neural results

In [ ]:
model_metrics = pd.DataFrame(rows)
predictions = pd.concat(predframes, ignore_index=True)
training_summary = pd.DataFrame(summary)

model_metrics.to_csv(
    OUTPUT_DIR / "paws_finetuned_all_metrics.csv",
    index=False
)
predictions.to_csv(
    OUTPUT_DIR / "paws_finetuned_predictions.csv",
    index=False
)
training_summary.to_csv(
    OUTPUT_DIR / "paws_training_summary.csv",
    index=False
)

display(
    model_metrics[
        model_metrics["split"].eq("test")
    ].sort_values("f1", ascending=False)
)

display(training_summary)


## 8. Lexical baselines

In [ ]:
def toks(s):
    return [t for t in clean(s).lower().split() if t]

def ngrams(s, n):
    s = clean(s).lower()
    if not s:
        return set()
    return {s} if len(s) < n else {s[i:i+n] for i in range(len(s)-n+1)}

def jac(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def dice(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    return 2 * len(a & b) / (len(a) + len(b)) if len(a) + len(b) else 0.0

def overlap(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))

def lex(s1, s2):
    t1, t2 = toks(s1), toks(s2)
    return {
        "levenshtein_similarity":
            float(Levenshtein.normalized_similarity(s1, s2)),
        "jaro_winkler_similarity":
            float(JaroWinkler.normalized_similarity(s1, s2)),
        "sequence_matcher_ratio":
            float(SequenceMatcher(None, s1, s2).ratio()),
        "token_jaccard": jac(t1, t2),
        "token_dice": dice(t1, t2),
        "token_overlap_coefficient": overlap(t1, t2),
        "char3_jaccard": jac(ngrams(s1,3), ngrams(s2,3)),
        "char4_jaccard": jac(ngrams(s1,4), ngrams(s2,4)),
    }

lexdf = pd.DataFrame(
    [lex(a,b) for a,b in zip(paws["sentence1_az"], paws["sentence2_az"])],
    index=paws.index
)

def choose_threshold(y, s):
    """Maximize dev F1, break ties by accuracy; sorted cumulative counts."""
    y = np.asarray(y, int); s = np.asarray(s, float)
    if len(y) == 0 or not np.isfinite(s).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Threshold selection requires nonempty finite scores and binary labels.")
    order = np.argsort(s, kind="stable"); sy = y[order]; ss = s[order]
    unique, starts = np.unique(ss, return_index=True)
    if len(unique) == 1:
        candidates = unique
        cuts = np.array([0])
    else:
        candidates = np.r_[unique[0]-1e-6, (unique[:-1]+unique[1:])/2, unique[-1]+1e-6]
        cuts = np.r_[0, starts[1:], len(y)]
    positives = np.r_[0, np.cumsum(sy)]
    tp = positives[-1] - positives[cuts]
    fp = len(y) - cuts - tp
    fn = positives[cuts]
    tn = cuts - fn
    denominator = 2*tp + fp + fn
    f1s = np.divide(2*tp, denominator, out=np.zeros(len(cuts), float), where=denominator != 0)
    accuracies = (tp+tn)/len(y)
    best_t, best_f, best_a = None, -1., -1.
    for t, f, a in zip(candidates, f1s, accuracies):
        if f > best_f or (np.isclose(f, best_f) and a > best_a):
            best_t, best_f, best_a = float(t), float(f), float(a)
    return best_t, best_f

def choose_thr(y, s):
    t, f = choose_threshold(y, s)
    return t, f, float(accuracy_score(y, np.asarray(s) >= t))

dev_idx = paws.index[paws["split"].eq("dev")]

lexrows = []
thresholds = []

for metric in lexdf.columns:
    t, dev_f1, dev_acc = choose_thr(
        paws.loc[dev_idx, "label"],
        lexdf.loc[dev_idx, metric]
    )

    thresholds.append({
        "metric": metric,
        "threshold": t,
        "selection_split": "dev",
        "selection_metric": "f1",
        "dev_f1": dev_f1,
        "dev_accuracy": dev_acc
    })

    for split in ["train", "dev", "test", "overall"]:
        idx = paws.index if split == "overall" else paws.index[paws["split"].eq(split)]

        lexrows.append({
            "model": "LEXICAL_BASELINE",
            "metric": metric,
            "split": split,
            "n": len(idx),
            **cls_metrics(
                paws.loc[idx, "label"],
                lexdf.loc[idx, metric],
                t
            )
        })

lex_metrics = pd.DataFrame(lexrows)
lex_thresholds = pd.DataFrame(thresholds)

lex_metrics.to_csv(
    OUTPUT_DIR / "paws_lexical_baseline_metrics.csv",
    index=False
)
lex_thresholds.to_csv(
    OUTPUT_DIR / "paws_lexical_thresholds.csv",
    index=False
)

lexout = paws[["sentence1_az", "sentence2_az", "label", "split"]].copy()
for c in lexdf.columns:
    lexout[c] = lexdf[c].values

lexout.to_csv(
    OUTPUT_DIR / "paws_lexical_scores.csv",
    index=False
)

display(
    lex_metrics[
        lex_metrics["split"].eq("test")
    ].sort_values("f1", ascending=False)
)


## 9. Check generated files

In [ ]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024**2)
        print(f"{size_mb:8.2f} MB  {p.relative_to(OUTPUT_DIR)}")


## 10. Download only the all-metrics CSV

In [ ]:
from IPython.display import FileLink

metrics_path = OUTPUT_DIR / "paws_finetuned_all_metrics.csv"
print(metrics_path)
FileLink(str(metrics_path))
